# Statistics for Genomics & GWAS
## Module 1 — Confidence Intervals

### Why this module matters

When you read a GWAS paper, every result looks like this:

> *"The G allele at rs4977574 was associated with increased risk of coronary artery disease (OR = 1.29, 95% CI: 1.22–1.37, p = 4.3 × 10⁻¹⁸)"*

To understand that sentence — and eventually to write sentences like it yourself — you need to deeply understand three things:
1. What a **confidence interval** actually means (this module)
2. What a **p-value** actually means (Module 2)
3. Why GWAS uses p < 5×10⁻⁸ instead of the usual 0.05 (Module 2)

Let's start from scratch.

---

### The core idea in one paragraph

Imagine you want to know the average LDL cholesterol level of all humans on Earth. You can't measure everyone — so you take a **sample** of, say, 30 people and compute their average. That sample average is your best **estimate** of the true population mean (μ). But it's just an estimate — a different sample of 30 people would give you a slightly different number.

A **confidence interval** (CI) quantifies that uncertainty. A 95% CI says: *"I'm reasonably confident the true mean lies somewhere in this range."* More precisely: if you repeated this sampling experiment 100 times and built a CI each time, about 95 of those intervals would contain the true μ. The other 5 would miss it entirely — and you'd never know which ones.

This is not the same as saying "there is a 95% probability the true mean is in this specific interval." The true mean either is or isn't in the interval — it's a fixed number. The 95% refers to the **procedure**, not to any single interval.


## 0. Setup

Run this cell first. We use only standard scientific Python libraries.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

# Fix the random seed so your results are reproducible.
# In real genomics work, always set a seed and report it.
np.random.seed(42)

# Clean plot style — similar to what you'd use in a publication
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print("Setup complete.")


## 1. Simulation: seeing confidence intervals in action

### The setup

Let's say the true mean LDL cholesterol in our population is μ = 130 mg/dL, with a standard deviation σ = 20 mg/dL. In real life we don't know μ — that's exactly what we're trying to estimate. But in a simulation we *do* know it, which lets us check whether our intervals are working correctly.

We will:
1. Draw 100 independent random samples of size n = 30
2. For each sample, compute the sample mean and a 95% CI
3. Count how many intervals contain the true μ = 130
4. Visualize all 100 intervals at once

If the math is correct, about 95 of them should capture μ. The remaining ~5 will miss — not because we made a mistake, but because that's what "95% confidence" means.


In [ ]:
# --- Parameters ---
TRUE_MU = 130      # true population mean (LDL in mg/dL) — we "pretend" not to know this
TRUE_SD = 20       # true population standard deviation
N = 30             # sample size for each experiment
N_EXPERIMENTS = 100
CONFIDENCE = 0.95  # try changing this to 0.80 or 0.99 and re-run

# --- Critical value ---
# z* is the number of standard errors we go left and right of the sample mean.
# For 95% confidence, we use the 97.5th percentile of the normal distribution.
# This gives us: mean ± 1.96 * SE
z_star = stats.norm.ppf((1 + CONFIDENCE) / 2)

# --- Standard error ---
# SE = SD / sqrt(n). It tells us how much the sample mean varies across samples.
# Larger n → smaller SE → narrower CI → more precise estimate.
se = TRUE_SD / np.sqrt(N)

print(f"Critical value z* for {CONFIDENCE*100:.0f}% CI: {z_star:.4f}")
print(f"Standard error SE = {TRUE_SD} / sqrt({N}) = {se:.2f} mg/dL")
print(f"Expected CI half-width: z* × SE = {z_star:.2f} × {se:.2f} = ±{z_star * se:.1f} mg/dL")
print()

# --- Run 100 experiments ---
results = []
for i in range(N_EXPERIMENTS):
    # Draw n random individuals from the population
    sample = np.random.normal(TRUE_MU, TRUE_SD, N)
    
    # Compute the sample mean — our estimate of μ
    mean = sample.mean()
    
    # Build the confidence interval: mean ± z* × SE
    lo = mean - z_star * se
    hi = mean + z_star * se
    
    # Check if this interval actually contains the true μ
    hit = (lo <= TRUE_MU <= hi)
    
    results.append({'experiment': i+1, 'mean': mean, 'lo': lo, 'hi': hi, 'hit': hit})

df = pd.DataFrame(results)
n_hits = df['hit'].sum()
n_miss = N_EXPERIMENTS - n_hits

print(f"Results after {N_EXPERIMENTS} experiments:")
print(f"  Intervals that captured μ = {TRUE_MU}:  {n_hits} ({n_hits}%)")
print(f"  Intervals that missed:                  {n_miss} ({n_miss}%)")
print(f"  Theoretical expectation:                {CONFIDENCE*100:.0f}%")


In [ ]:
# --- Visualization ---
# Each horizontal line is one experiment.
# Green = the interval captured the true μ.
# Red   = the interval missed the true μ.
# The dashed vertical line is the true μ = 130.

fig, ax = plt.subplots(figsize=(12, 9))

for _, row in df.iterrows():
    color = '#1D9E75' if row['hit'] else '#D85A30'
    # Plot the confidence interval as a horizontal line
    ax.hlines(row['experiment'], row['lo'], row['hi'],
              colors=color, linewidth=1.4, alpha=0.75)
    # Plot the sample mean as a dot
    ax.plot(row['mean'], row['experiment'], 'o',
            color=color, markersize=2.5, alpha=0.75)

# True population mean
ax.axvline(TRUE_MU, color='#2C2C2A', linewidth=1.5,
           linestyle='--', label=f'True μ = {TRUE_MU}')

hit_patch  = mpatches.Patch(color='#1D9E75', label=f'Captured μ ({n_hits})')
miss_patch = mpatches.Patch(color='#D85A30', label=f'Missed μ ({n_miss})')
ax.legend(handles=[hit_patch, miss_patch], loc='lower right', fontsize=10)

ax.set_xlabel('LDL cholesterol (mg/dL)')
ax.set_ylabel('Experiment number')
ax.set_title(
    f'{CONFIDENCE*100:.0f}% Confidence Intervals — {N_EXPERIMENTS} simulated samples (n={N})\n'
    f'{n_hits} of {N_EXPERIMENTS} intervals captured the true mean ({n_hits}%)',
    fontsize=12
)
plt.tight_layout()
plt.show()

print("Notice: the intervals that missed are not 'wrong' — they are the expected ~5%.")
print("In real research you never know which interval you're looking at.")


### What to try

**Experiment 1:** Change `CONFIDENCE = 0.95` to `0.80` and re-run.  
You should see wider green bars disappear — more misses, but narrower intervals.  
That's the trade-off: more confidence → wider interval → less useful for pinpointing an effect.

**Experiment 2:** Change `N = 30` to `N = 5` and re-run.  
The intervals become very wide. With only 5 people, your sample mean is noisy —  
you need a wide net to be confident it contains the true value.  
This is exactly why GWAS studies need hundreds of thousands of participants.

**Experiment 3:** Change `N = 30` to `N = 500`.  
Intervals shrink dramatically. You can estimate μ precisely with large samples.


## 2. The formula — building CI from scratch

### When you know σ (population standard deviation)

$$\bar{x} \pm z^* \cdot \frac{\sigma}{\sqrt{n}}$$

This is the textbook formula. The problem: in real research **you never know σ**.  
You only have your sample.

### When you don't know σ (almost always)

Replace z* with **t*** from the t-distribution, and estimate σ using the sample standard deviation s:

$$\bar{x} \pm t^* \cdot \frac{s}{\sqrt{n}}$$

The t-distribution has heavier tails than the normal distribution — it's more conservative when sample sizes are small. As n grows, t* converges to z*. For n > 30, the difference is negligible.

The term $\frac{s}{\sqrt{n}}$ is called the **standard error of the mean (SE)**. It measures how much the sample mean varies across repeated samples.


In [ ]:
def confidence_interval(data, confidence=0.95):
    """
    Compute a confidence interval for the mean using the t-distribution.
    
    We use t (not z) because we estimate σ from the data.
    The t-distribution has heavier tails than normal — it's more 
    conservative with small samples, which is the right behavior.
    
    Parameters
    ----------
    data       : array-like of observed values
    confidence : desired confidence level (default 0.95)
    
    Returns
    -------
    mean, lower_bound, upper_bound, margin_of_error
    """
    n    = len(data)
    mean = np.mean(data)
    s    = np.std(data, ddof=1)     # sample std dev (ddof=1 = unbiased estimate)
    se   = s / np.sqrt(n)           # standard error of the mean
    
    # Degrees of freedom for t-distribution = n - 1
    # As n → ∞, t* → z* (the normal critical value)
    df     = n - 1
    t_star = stats.t.ppf((1 + confidence) / 2, df=df)
    
    moe = t_star * se               # margin of error
    return mean, mean - moe, mean + moe, moe


# --- Example: 30 patients, LDL measurements ---
sample = np.random.normal(TRUE_MU, TRUE_SD, 30)

mean, lo, hi, moe = confidence_interval(sample, confidence=0.95)

print("Sample of 30 LDL measurements:")
print(f"  Sample mean:       {mean:.1f} mg/dL")
print(f"  Sample std dev:    {np.std(sample, ddof=1):.1f} mg/dL")
print(f"  Standard error:    {np.std(sample, ddof=1)/np.sqrt(30):.2f} mg/dL")
print(f"  95% CI:            [{lo:.1f}, {hi:.1f}]")
print(f"  Margin of error:   ±{moe:.1f} mg/dL")
print(f"  True μ = {TRUE_MU} is {'inside the CI ✓' if lo <= TRUE_MU <= hi else 'outside the CI ✗'}")
print()

# Compare z* vs t* for different sample sizes
print("How t* approaches z* as n grows:")
print(f"  z* (normal, n→∞) = {stats.norm.ppf(0.975):.4f}")
for n in [5, 10, 20, 50, 100, 1000]:
    t = stats.t.ppf(0.975, df=n-1)
    print(f"  t* (n={n:5d}, df={n-1:4d}) = {t:.4f}")


## 3. How sample size n affects CI width

### The key relationship

$$\text{CI width} = 2 \cdot t^* \cdot \frac{s}{\sqrt{n}}$$

Width shrinks proportionally to $\frac{1}{\sqrt{n}}$.  
To halve the CI width, you need **4× more participants**.  
To make it 10× narrower, you need **100× more participants**.

This is why GWAS studies invest enormously in sample size. A typical GWAS for a complex disease like coronary artery disease uses **500,000 to 1,000,000 participants**. With smaller samples, the CIs around effect sizes would be so wide that the results would be meaningless.

A SNP with OR = 1.05 (a small but real effect) might have a CI of [0.95, 1.15] in a study of 10,000 people — that CI crosses 1.0, so you can't distinguish it from noise. With 500,000 people, the same OR might have CI = [1.02, 1.08] — clearly above 1.0, clearly a real signal.


In [ ]:
sample_sizes = [5, 10, 20, 50, 100, 500, 1000, 5000]

# Compute CI width for each sample size (averaged over 200 simulations)
mean_widths = []
for n in sample_sizes:
    widths = []
    for _ in range(200):
        sample = np.random.normal(TRUE_MU, TRUE_SD, n)
        _, lo, hi, _ = confidence_interval(sample)
        widths.append(hi - lo)
    mean_widths.append(np.mean(widths))

# --- Figure with two panels ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: CI width vs n
axes[0].plot(sample_sizes, mean_widths, 'o-',
             color='#534AB7', linewidth=2, markersize=7, label='Simulated width')

# Overlay the theoretical curve: width ≈ 2 * z* * sigma / sqrt(n)
n_theory = np.linspace(5, 5000, 500)
w_theory = 2 * 1.96 * TRUE_SD / np.sqrt(n_theory)
axes[0].plot(n_theory, w_theory, '--',
             color='#1D9E75', alpha=0.7, label='Theoretical (z*)')

axes[0].set_xscale('log')
axes[0].set_xlabel('Sample size n (log scale)')
axes[0].set_ylabel('Average 95% CI width (mg/dL)')
axes[0].set_title('CI width shrinks with sample size
(log scale on x-axis)')
axes[0].legend()

# Annotation: doubling n doesn't halve the width — squaring does
axes[0].annotate('Width ∝ 1/√n
(4× more people → 2× narrower)',
                 xy=(50, 2*1.96*TRUE_SD/np.sqrt(50)),
                 xytext=(15, 12),
                 arrowprops=dict(arrowstyle='->', color='gray'),
                 fontsize=9, color='gray')

# Panel 2: Visual comparison of CIs for each n
for i, (n, w) in enumerate(zip(sample_sizes, mean_widths)):
    sample = np.random.normal(TRUE_MU, TRUE_SD, n)
    m, lo, hi, _ = confidence_interval(sample)
    hit   = lo <= TRUE_MU <= hi
    color = '#1D9E75' if hit else '#D85A30'
    axes[1].hlines(i, lo, hi, colors=color, linewidth=4, alpha=0.7)
    axes[1].plot(m, i, 'D', color=color, markersize=6, zorder=5)

axes[1].axvline(TRUE_MU, color='#2C2C2A', linestyle='--', linewidth=1.5,
                label=f'True μ = {TRUE_MU}')
axes[1].set_yticks(range(len(sample_sizes)))
axes[1].set_yticklabels([f'n = {n}' for n in sample_sizes])
axes[1].set_xlabel('LDL cholesterol (mg/dL)')
axes[1].set_title('One CI per sample size
(same true μ = 130 in all cases)')
axes[1].legend(fontsize=9)

plt.suptitle('Effect of sample size on confidence interval width', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Summary table:")
print(f"{'n':>6}  {'CI width':>10}  {'Relative to n=5':>16}")
base = mean_widths[0]
for n, w in zip(sample_sizes, mean_widths):
    print(f"{n:>6}  {w:>8.1f}    {w/base:>14.2f}×")


## 4. CI in GWAS — effect size and Odds Ratio

### From means to odds ratios

In GWAS for a binary outcome (disease: yes/no), the effect of a SNP is expressed as an **Odds Ratio (OR)** rather than a mean difference.

**Odds Ratio interpretation:**
- OR = 1.00 → no effect (the allele is unrelated to disease risk)
- OR = 1.30 → carrying the risk allele increases odds of disease by 30%
- OR = 0.85 → carrying the allele is *protective* — it reduces odds by 15%
- **A CI that crosses 1.0 means the result is not statistically significant** — we can't rule out "no effect"

### How OR is computed (simplified)

For a SNP with two alleles A and G:

|            | Has disease | No disease |
|------------|-------------|------------|
| Carries G  | a           | b          |
| No G       | c           | d          |

$$OR = \frac{a/b}{c/d} = \frac{a \cdot d}{b \cdot c}$$

The CI for OR is computed on the log scale (log-OR is approximately normally distributed), then exponentiated back.

### The forest plot

In GWAS papers, results are shown as a **forest plot**: each SNP gets a row, with a diamond at the OR and a horizontal line showing the CI. If the line doesn't cross 1.0 → the association is statistically significant.


In [ ]:
# Simulated GWAS results for 8 SNPs associated with coronary artery disease (CAD).
# These are stylized but realistic — based on the kind of results you'd see
# in a real CAD GWAS like the CARDIoGRAMplusC4D consortium paper.

gwas_results = pd.DataFrame({
    'snp':   ['rs1234',  'rs5678', 'rs9012', 'rs3456',
              'rs4977574','rs2345', 'rs6789', 'rs0123'],
    'chrom': ['chr1',    'chr3',   'chr6',   'chr9',
              'chr9',    'chr12',  'chr15',  'chr19'],
    'gene':  ['PCSK9',   'LDLR',   'APOE',   'ABO',
              'CDKN2B',  'LPL',    'CETP',   'FTO'],
    'OR':    [1.32,  1.18,  0.85,  1.09,   1.27,  1.14,  0.91,  1.06],
    'OR_lo': [1.24,  1.08,  0.78,  0.98,   1.19,  1.07,  0.85,  0.99],
    'OR_hi': [1.41,  1.29,  0.93,  1.21,   1.36,  1.22,  0.97,  1.13],
    'p_val': [2.1e-18, 3.4e-9, 8.7e-11, 0.12,
              4.5e-15,  6.2e-7, 1.3e-4,  0.08],
    'n_samples': [350000, 350000, 350000, 350000,
                  350000, 350000, 350000, 350000],
})

# Standard GWAS significance threshold (we'll explain why in Module 2)
gwas_results['significant'] = gwas_results['p_val'] < 5e-8

# CIs are computed on the log(OR) scale, then plotted with OR labels on the x-axis
gwas_results['log_OR'] = np.log(gwas_results['OR'])
gwas_results['log_lo'] = np.log(gwas_results['OR_lo'])
gwas_results['log_hi'] = np.log(gwas_results['OR_hi'])

print("GWAS results table:")
display_cols = ['snp', 'gene', 'OR', 'OR_lo', 'OR_hi', 'p_val', 'significant']
print(gwas_results[display_cols].to_string(index=False))


In [ ]:
# --- Forest plot ---
# This is a standard figure type in GWAS papers and meta-analyses.
# Read it like this:
#   - Diamond = OR (effect size estimate)
#   - Horizontal line = 95% CI
#   - Vertical dashed line at 1.0 = "no effect" null hypothesis
#   - If the CI line doesn't touch the dashed line → statistically significant

fig, ax = plt.subplots(figsize=(11, 7))

n_snps = len(gwas_results)

for i, row in gwas_results.iterrows():
    y     = n_snps - 1 - i          # plot top-to-bottom
    color = '#534AB7' if row['significant'] else '#B4B2A9'
    alpha = 0.85

    # CI line
    ax.hlines(y, row['log_lo'], row['log_hi'],
              colors=color, linewidth=2.5, alpha=alpha)
    # OR diamond
    ax.plot(row['log_OR'], y, 'D',
            color=color, markersize=8, alpha=alpha, zorder=5)

    # SNP and gene label on the left
    ax.text(-0.52, y, row['snp'],  ha='right', va='center',
            fontsize=10, color='#2C2C2A')
    ax.text(-0.92, y, row['gene'], ha='right', va='center',
            fontsize=10, color='#888780', style='italic')

    # OR value on the right with CI
    label = f"{row['OR']:.2f} [{row['OR_lo']:.2f}–{row['OR_hi']:.2f}]"
    ax.text(ax.get_xlim()[1] if ax.get_xlim()[1] > 0.5 else 0.5,
            y, label, ha='left', va='center', fontsize=9, color=color)

# Null line (OR = 1.0, log(OR) = 0)
ax.axvline(0, color='#444', linewidth=1, linestyle='--', alpha=0.6,
           label='OR = 1.0 (no effect)')

# X-axis: show OR values instead of log(OR)
or_ticks = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]
ax.set_xticks(np.log(or_ticks))
ax.set_xticklabels([str(v) for v in or_ticks])
ax.set_xlim(-1.05, 0.65)

ax.set_yticks([])
ax.set_xlabel('Odds Ratio (OR)          ← protective effect  |  risk effect →',
              fontsize=10)
ax.set_title('Forest plot — effect size and 95% CI for each SNP
'
             '(diamond = OR estimate, line = 95% CI, purple = p < 5×10⁻⁸)',
             fontsize=12)

sig_patch = mpatches.Patch(color='#534AB7', label='Significant (p < 5×10⁻⁸)')
ns_patch  = mpatches.Patch(color='#B4B2A9', label='Not significant')
ax.legend(handles=[sig_patch, ns_patch], loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# --- Interpreting the results ---
print("=== Interpretation guide ===\n")

sig = gwas_results[gwas_results['significant']]
ns  = gwas_results[~gwas_results['significant']]

print("SIGNIFICANT SNPs (CI does not cross OR=1.0, p < 5×10⁻⁸):")
for _, r in sig.iterrows():
    direction = "increases risk" if r['OR'] > 1 else "protective"
    print(f"  {r['snp']} ({r['gene']:6s}): OR={r['OR']:.2f} [{r['OR_lo']:.2f}–{r['OR_hi']:.2f}]"
          f"  →  {direction}  (p={r['p_val']:.1e})")

print()
print("NON-SIGNIFICANT SNPs (CI crosses OR=1.0, p ≥ 0.05):")
for _, r in ns.iterrows():
    print(f"  {r['snp']} ({r['gene']:6s}): OR={r['OR']:.2f} [{r['OR_lo']:.2f}–{r['OR_hi']:.2f}]"
          f"  →  CI crosses 1.0, cannot rule out no effect  (p={r['p_val']:.2f})")

print()
print("Key insight:")
print("  rs3456 has OR=1.09, which sounds like a real effect.")
print("  But the CI [0.98–1.21] crosses 1.0, so we cannot reject the null hypothesis.")
print("  With a larger sample, the CI would narrow and might become significant — or stay NS.")
print("  This is why GWAS needs massive sample sizes.")


## 5. Exercises

### Exercise 1 — Understand the definition
Change `CONFIDENCE = 0.95` to `0.99` in Section 1 and re-run the simulation.

- Does the number of misses decrease?
- Do the intervals get wider or narrower? Why?
- What is the trade-off between confidence level and interval width?

### Exercise 2 — Implement bootstrap CI
The formula-based CI assumes the data is approximately normally distributed.  
A **bootstrap CI** makes no such assumption — it's more robust for small samples or skewed data. This is commonly used in genomics.

```python
def bootstrap_ci(data, confidence=0.95, n_bootstrap=2000):
    """
    Estimate a CI by resampling the data with replacement.
    
    For each bootstrap sample:
      1. Draw n values from data WITH replacement (some appear twice, some not at all)
      2. Compute the mean of this bootstrap sample
    After n_bootstrap iterations, the CI is the (alpha/2) and (1-alpha/2) percentiles
    of all bootstrap means.
    """
    alpha       = 1 - confidence
    boot_means  = []
    n           = len(data)
    
    for _ in range(n_bootstrap):
        # YOUR CODE HERE
        # Hint: use np.random.choice(data, size=n, replace=True)
        pass
    
    lo = np.percentile(boot_means, 100 * alpha / 2)
    hi = np.percentile(boot_means, 100 * (1 - alpha / 2))
    return np.mean(data), lo, hi

# Test it:
sample = np.random.normal(130, 20, 30)
mean_f, lo_f, hi_f, _ = confidence_interval(sample)
mean_b, lo_b, hi_b    = bootstrap_ci(sample)

print(f"Formula-based CI: [{lo_f:.1f}, {hi_f:.1f}]")
print(f"Bootstrap CI:     [{lo_b:.1f}, {hi_b:.1f}]")
print("They should be very similar for normally distributed data.")
```

### Exercise 3 — Interpret the forest plot
Look at the forest plot in Section 4 and answer:
1. Which SNP has the strongest *protective* effect? How do you know?
2. Which SNP has the widest CI? What does that mean about our certainty of its effect?
3. rs4977574 in *CDKN2B* has one of the smallest p-values. What is its OR, and is the effect biologically large or small?
4. Why do all SNPs have the same sample size but different CI widths?  
   *(Hint: OR and its variance depend on allele frequency, not just n)*

### Exercise 4 — Preview of Module 2
Think about this before the next module:

In GWAS we test ~1,000,000 SNPs simultaneously.  
For each SNP we ask: "is this allele associated with disease?"  
We declare a result significant if p < 0.05 (the standard threshold in other sciences).

**Question:** If none of the 1,000,000 SNPs had any real effect on disease,  
how many would we expect to pass the p < 0.05 threshold purely by chance?

Write your answer here, then check it in Module 2.

---
## Next: Module 2 — p-value and Multiple Testing

We will show *exactly* why GWAS uses p < 5×10⁻⁸ instead of 0.05,  
simulate the false discovery explosion that happens without correction,  
and implement Bonferroni correction from scratch.

The answer to Exercise 4 is the starting point.
